In [1]:
#Based on Modern Robotics Library 
import numpy as np 
import modern_robotics as mr

In [2]:
#This function will extract components from the calculated matrices and store them in a list
#This function takes in a list of matrices after running trajectory and the gripper status
# outputs a list consisting of the components and status
def extract_components_from_matrix(trajectory_matrices, gripper_status):
    #init row list
    row_list = []
    for M in trajectory_matrices:
        #extracting info from each matrix M in a list of trajectory matrices
        rows = M[0:3, 0:3].flatten() 
        pos = M[0:3, 3]

        #row_format is in form of [r11,r12,r13,r21,r22,r23,r31,r32,r33,px,py,pz,gripper]
        #appending row format into row list
        row_format = [rows[0], rows[1], rows[2], rows[3], rows[4], rows[5], rows[6], rows[7], rows[8], pos[0], pos[1], pos[2], gripper_status]
        row_list.append(row_format)
    return row_list

In [3]:
#Trajectory Generation Code generates trajectory for the robot 
#inputs: Tse, Tsc_init , Tsc_final, Tce_grasp, Tce_standoff, k 
#all matrices need to be in the form of np.array([[...], [...], ... ])
#outputs a list of all trajectory waypoints (for saving results as .csv file, need to type np.savetxt("trajectory.csv", TrajectoryGenerator(...), delimiter=","))
def TrajectoryGenerator(Tse, Tsc_init, Tsc_final, Tce_grasp, Tce_standoff, k):

    #init waypoint list
    waypoint_list = []

    #converting to world frame first for end effector (need to double check if this is correct)
    Tse_grasp_init = Tsc_init @ Tce_grasp
    Tse_grasp_final = Tsc_final @ Tce_grasp
    
    Tse_standoff_init = Tsc_init @ Tce_standoff
    Tse_standoff_final = Tsc_final @ Tce_standoff
    
    #gripper status
    gripper_status =  0 #0 for open, 1 for closed

    #trajectory 1: using CartesianTrajectory to go from Tse to Tse_standoff initial
    #extracting trajectory from matrices and appending to waypoint list
    Tf = 2
    number_of_points = Tf*k/0.01
    trajectory_1 = mr.CartesianTrajectory(Tse, Tse_standoff_init,Tf,number_of_points,5)
    gripper_status = 0
    extraction_1 = extract_components_from_matrix(trajectory_1, gripper_status)
    waypoint_list.extend(extraction_1)


    #trajectory 2: using CartesianTrajectory to go from Tse_standoff initial to Tse_grasp initial 
    #extracting trajectory from matrices and appending to waypoint list
    trajectory_2 = mr.CartesianTrajectory(Tse_standoff_init, Tse_grasp_init,Tf,number_of_points,5)
    gripper_status = 0
    extraction_2 = extract_components_from_matrix(trajectory_2, gripper_status)
    waypoint_list.extend(extraction_2)

    #trajectory 3: closing gripper 
    #extracting trajectory from matrices and appending to waypoint list
    trajectory_3 =  mr.CartesianTrajectory(Tse_grasp_init, Tse_grasp_init, Tf,number_of_points,5)
    gripper_status = 1 
    extraction_3 = extract_components_from_matrix(trajectory_3, gripper_status)
    waypoint_list.extend(extraction_3)

    #trajectory 4: using CartesianTrajectory to go to Tse_standoff initial from ground 
    #extracting trajectory from matrices and appending to waypoint list
    trajectory_4 = mr.CartesianTrajectory(Tse_grasp_init, Tse_standoff_init, Tf,number_of_points,5)
    gripper_status = 1
    extraction_4 = extract_components_from_matrix(trajectory_4, gripper_status)
    waypoint_list.extend(extraction_4)

    #trajectory 5: using CartesianTrajectory to go to Tse_standoff final from Tse_standoff initial
    #extracting trajectory from matrices and appending to waypoint list 
    trajectory_5 = mr.CartesianTrajectory(Tse_standoff_init,Tse_standoff_final, Tf,number_of_points,5)
    gripper_status = 1
    extraction_5 = extract_components_from_matrix(trajectory_5, gripper_status)
    waypoint_list.extend(extraction_5)

    #trajectory 6: using CartesianTrajectory to go to the Tse_grasp final of the robot from Tse_standoff final 
    #extracting trajectory from matrices and appending to waypoint list
    trajectory_6 = mr.CartesianTrajectory(Tse_standoff_final, Tse_grasp_final,Tf, number_of_points,5)
    gripper_status = 1 
    extraction_6 = extract_components_from_matrix(trajectory_6, gripper_status)
    waypoint_list.extend(extraction_6)
    
    #trajectory 7: opening gripper
    #extracting trajectory from matrices and appending to waypoint list 
    trajectory_7 = mr.CartesianTrajectory(Tse_grasp_final, Tse_grasp_final, Tf, number_of_points,5)
    gripper_status = 0 
    extraction_7 = extract_components_from_matrix(trajectory_7, gripper_status)
    waypoint_list.extend(extraction_7)

    #trajectory 8: using CartesianTrajectory to finish at Tse_standoff final 
    #extracting trajectory from matrices and appending to waypoint list
    trajectory_8 = mr.CartesianTrajectory(Tse_grasp_final, Tse_standoff_final, Tf, number_of_points, 5)
    gripper_status = 0 
    extraction_8 = extract_components_from_matrix(trajectory_8, gripper_status)
    waypoint_list.extend(extraction_8)

    return waypoint_list 


In [4]:
#testing 
Tse_init = np.array([[0,0,1,0], [0,1,0,0], [-1,0,0,0.4], [0,0,0,1]])
Tsc_init = np.array([[1,0,0,1], [0,1,0,0], [0, 0, 1, 0.025], [0,0,0,1]])
Tsc_final = np.array([[0, 1, 0, 0],[-1, 0, 0, -1],[0, 0, 1, 0.025],[0, 0, 0, 1]])
Tce_grasp = np.array([[-1, 0, 0, 0],[0, 1, 0, 0],[0, 0, -1, 0],[0, 0, 0, 1]])
Tce_standoff = np.array([[-1, 0, 0, 0],[0, 1, 0, 0],[0, 0, -1, 0.1],[0, 0, 0, 1]])
result = TrajectoryGenerator(Tse_init, Tsc_init, Tsc_final, Tce_grasp, Tce_standoff, 1)
np.savetxt("trajectory.csv", result, delimiter=",")
